In [ ]:
import re, json
from typing import List, Tuple, Optional, Any
from pydantic import BaseModel, Field
from crewai import Agent, Task, Crew, Process, LLM
import time

# TUTORING FLOW

input:
- pseoudocode
- problem
- context solution
- general rubrication

agen:
- scoring supervisior
- sub-agent 1: style checker agent
- sub-agent 2: logic checker agent

output:
```json
{
  "score":"value_scorenya",
  "correct":"true/false",
  "summary":"some_summary_here",
  "Misconceptions":"misconceptionsnya apa",
  "pseudocode":"pseudocode_siswa"
}
```

Possible Prompt for Each Agent:
- zero-shot
- few-shot
- chain-of-thought

TO DO Voice:
- Mendesain multi-agent system dengan 3 agen di dalamnya yaitu Style Checker, Logic Checker, dan Scoring Supervisor. Gunakan iterasi ganjil dalam interaksi antar agent seperti 1,3,5,dst.
- Eksplorasi berbagai LLM (free resource and can be fine tuned) yang bisa digunakan ke dalam agent.
- Gunakan prompting zero-shot atau COT pada Style Checker dan Logic Checker (boleh prompting lain juga).
- Tech-Stacknya bisa pakai Langchain / Langgraph

In [ ]:
# Pastikan import ini sesuai SDK Crew/Agent/LLM yang kamu pakai
# from crewai_sdk import LLM, Agent, Task, Crew, Process  # contoh nama modul

# -------------------------
#  Helper / KB utilities
# -------------------------
def read_text_file(file_path: str) -> str:
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        return f"Error: File {file_path} tidak ditemukan. Pastikan file ada di folder yang sama."

def chunk_pseudocode_kb(raw_text: str) -> dict:
    sections = re.split(r'\n(?=## \d+-)', raw_text)
    chunks = {}
    for section in sections:
        match = re.match(r'## \d+-(.+)', section.strip())
        if match:
            key = match.group(1).strip().lower().replace('-', '_')
            chunks[key] = section.strip()
        else:
            if section.strip():
                chunks['intro'] = section.strip()
    return chunks

def chunk_misconceptions_kb(raw_text: str) -> dict:
    categories = {}
    current_cat = None
    for line in raw_text.splitlines():
        header_match = re.match(r'^## (.+)', line.strip())
        if header_match:
            current_cat = header_match.group(1).strip()
            categories[current_cat] = []
        elif current_cat and line.strip().startswith('- '):
            categories[current_cat].append(line.strip())
    return categories

def strip_golang_examples(chunk_text: str) -> str:
    lines = chunk_text.split('\n')
    result = []
    skip = False
    for line in lines:
        if line.strip().startswith('``` go') or line.strip().startswith('```go') or \
           line.strip().startswith('``` plaintext') or line.strip().startswith('```plaintext'):
            skip = True
            continue
        if skip and line.strip() == '```':
            skip = False
            continue
        if skip:
            continue
        if line.strip().startswith('**Output:**'):
            continue
        result.append(line)
    return '\n'.join(result)

# (Re-use select_pseudocode_chunks & select_misconception_chunks from your version)
def select_pseudocode_chunks(chunks, pseudocode_siswa, max_chars=4000):
    code = pseudocode_siswa.lower()
    always_relevant = ['intro', 'variabel_dan_tipe_data', 'operator']
    keyword_map = {
        'output':       'output',
        'input':        'input_dan_output',
        'constant':     'konstanta',
        'for ':         'perulangan',
        'while ':       'perulangan',
        'repeat':       'perulangan',
        'endfor':       'perulangan',
        'endwhile':     'perulangan',
        'if ':          'percabangan',
        'else':         'percabangan',
        'endif':        'percabangan',
    }
    selected_keys = set(always_relevant)
    for kw, chunk_name in keyword_map.items():
        if kw in code:
            selected_keys.add(chunk_name)
    if 'perulangan' in selected_keys and 'percabangan' in selected_keys:
        selected_keys.add('perulangan_dan_percabangan')
    ordered_keys = [k for k in always_relevant if k in chunks]
    ordered_keys += [k for k in selected_keys if k not in always_relevant and k in chunks]
    parts = []
    total_chars = 0
    for key in ordered_keys:
        cleaned = strip_golang_examples(chunks[key])
        if total_chars + len(cleaned) > max_chars:
            remaining = max_chars - total_chars
            if remaining > 200:
                parts.append(cleaned[:remaining] + "\n...[dipotong]")
            break
        parts.append(cleaned)
        total_chars += len(cleaned)
    return "\n\n---\n\n".join(parts)

def select_misconception_chunks(categories, pseudocode_siswa):
    code = pseudocode_siswa.lower()
    loop_misconceptions = {
        'While demon (WD)', 'Intentional bug (IB)', 'Mix of intentional bug and while demon (IBxWD)',
        'Conditional loop as conditional statement without alternative (WhileIf)', 'Executed once (EO)',
    }
    conditional_misconceptions = {'Conditional statement without alternative as conditional loop (IfWhile)', 'Drop through error (DT)'}
    loop_errors = {'Full program as loop (SNIP)', 'Execute n statement (EXN)', 'Miscounting loop (Loop+ atau Loop-)'}
    nesting_errors = {'Ignore nesting 1 (IN1)', 'Ignore nesting 2 (IN2)', 'Ignore outer loop (IOL)', 'Multiply counter (MC)'}
    has_loop = any(kw in code for kw in ['while ', 'for ', 'repeat', 'endwhile', 'endfor'])
    has_conditional = any(kw in code for kw in ['if ', 'else', 'endif'])
    has_nested = code.count('for ') > 1 or code.count('while ') > 1 or ('for ' in code and 'while ' in code)
    relevant_names = set()
    if has_loop:
        relevant_names |= loop_misconceptions | loop_errors
    if has_conditional:
        relevant_names |= conditional_misconceptions
    if has_nested:
        relevant_names |= nesting_errors
    result_parts = []
    for cat, items in categories.items():
        if 'ketidaktelitian' in cat.lower() or 'carelessness' in cat.lower():
            result_parts.append(f"## {cat}\n" + "\n".join(items))
            continue
        filtered = [item for item in items if any(name.lower() in item.lower() for name in relevant_names)]
        if filtered:
            result_parts.append(f"## {cat}\n" + "\n".join(filtered))
    if not result_parts:
        return "\n\n".join(f"## {cat}\n" + "\n".join(items) for cat, items in categories.items())
    return "\n\n".join(result_parts)

# -------------------------
#  Validation helpers
# -------------------------
def validate_json_response(text: str, required_keys: List[str]) -> Tuple[bool, Optional[str], Optional[Any]]:
    """
    Validate that `text` is a JSON object and contains required_keys.
    Returns (ok, error_reason, parsed_obj)
    """
    try:
        obj = json.loads(text)
    except Exception as e:
        return False, f"not_json: {e}", None
    if not isinstance(obj, dict):
        return False, "not_object", obj
    missing = [k for k in required_keys if k not in obj]
    if missing:
        return False, f"missing_keys: {missing}", obj
    return True, None, obj

# -------------------------
#  Pydantic schema for final output (unchanged)
# -------------------------
class AssessmentResult(BaseModel):
    score: int = Field(..., description="Nilai akhir siswa (integer 0-100)")
    correct: bool = Field(..., description="True jika logika benar, False jika salah")
    summary: str = Field(..., description="Ringkasan naratif penilaian")
    misconceptions: List[str] = Field(..., description="List nama miskonsepsi yang ditemukan (sesuai daftar referensi)")
    pseudocode: str = Field(..., description="Pseudocode ASLI siswa tanpa modifikasi")

# -------------------------
#  LLM + Agent config (example)
# -------------------------
llm = LLM(
    model="ollama/llama3.1:8b",
    base_url="http://localhost:11434",
    temperature=0.0
)

# -------------------------
#  Prepare chunked KB
# -------------------------
_raw_pseudocode = read_text_file("/home/ilham/Documents/python/crewai-vs-langgraph/doc/Pseudocode dan Golang Dasar.md")
_raw_misconceptions = read_text_file("/home/ilham/Documents/python/crewai-vs-langgraph/doc/List of misconceptions.md")
pseudocode_chunks = chunk_pseudocode_kb(_raw_pseudocode)
misconception_chunks = chunk_misconceptions_kb(_raw_misconceptions)

# -------------------------
#  Define agents (with strict JSON contract in their task backstories/prompts)
# -------------------------
# Few-shot JSON examples to force format
STYLE_JSON_EXAMPLE = """
EXAMPLE 1 (no violations):
Input: pseudocode that is correct
Output (JSON):
{"violations": [], "summary": "OK", "score_deduction": 0}

EXAMPLE 2 (with violation):
Input: pseudocode uses '=' instead of '<-'
Output (JSON):
{"violations": [{"type":"assignment_syntax","message":"Uses '=' instead of '<-'", "line": 4, "penalty": 5}], "summary":"Found assignment syntax errors", "score_deduction": 5}
"""

LOGIC_JSON_EXAMPLE = """
EXAMPLE 1 (no misconceptions):
Output (JSON):
{"misconceptions_detected": [], "summary": "No misconceptions", "penalty": 0}

EXAMPLE 2 (found WD):
Output (JSON):
{"misconceptions_detected": ["While demon (WD)"], "summary": "While loop uses premature break pattern", "penalty": 10}
"""

style_checker = Agent(
    role="Style & Syntax Auditor",
    goal="Memvalidasi kepatuhan pseudocode terhadap standar penulisan yang baku and RETURN JSON only.",
    backstory=(
        "You are an auditor that MUST RETURN a single JSON object (no extra prose). "
        "Schema: {violations: list, summary: str, score_deduction: int}. "
        "Each violation item: {type, message, line?, penalty}. "
        "Return ONLY JSON. " + STYLE_JSON_EXAMPLE
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=1
)

logic_checker = Agent(
    role="Logic & Misconception Analyst", 
    goal="Analyze algorithm correctness and detect misconceptions. RETURN JSON only.",
    backstory=(
        "You are a logic analyst that MUST RETURN a single JSON object (no extra prose). "
        "Schema: {misconceptions_detected: list, summary: str, penalty: int}. "
        "Return ONLY JSON. " + LOGIC_JSON_EXAMPLE
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=1
)

scoring_supervisor = Agent(
    role="Scoring Supervisor",
    goal="Aggregate sub-agent JSON outputs and produce final JSON conforming to AssessmentResult.",
    backstory=(
        "You are an aggregator. You MUST NOT invent violations. Use only data from sub-agents' JSON outputs. "
        "Return final Pydantic-valid JSON (AssessmentResult). "
        "Calculate score as: 100 - style_deductions - logic_penalties. "
        "Set correct=False if any misconceptions found or logic errors detected."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,  # Changed to False
    memory=False,  # Simplified
    max_iter=1
)
# -------------------------
#  Safe step_callback
# -------------------------
def step_callback(step):
    try:
        if isinstance(step, dict):
            agent = step.get("agent_name", "UnknownAgent")
            task = step.get("task_description", "")
            count = step.get("step_count", "?")
            print(f"↺ Iterasi {count}: {agent} - {task[:50]}...")
        else:
            # Generic object (AgentAction / AgentFinish / rich object)
            tname = type(step).__name__
            # Try to print some useful attributes if present
            name = getattr(step, "agent_name", None) or getattr(step, "agent", None) or ""
            brief = ""
            if hasattr(step, "log") and isinstance(getattr(step, "log"), str):
                brief = getattr(step, "log")[:80]
            print(f"↺ {tname} {('from '+str(name))}: {brief}")
    except Exception as e:
        print(f"! step_callback error: {e}")

# -------------------------
#  Tasks with contracts (note placeholders will be filled at runtime)
# -------------------------
# We'll build the Task.description at runtime after selecting chunks.

# -------------------------
#  Utility: run crew with retries on parsing failure
# -------------------------
def run_evaluation_with_retries(crew: Any, inputs: dict, max_attempts: int = 2) -> AssessmentResult:
    for attempt in range(max_attempts):
        try:
            print(f"\n>> RUN ATTEMPT {attempt + 1}/{max_attempts}")
            result = crew.kickoff(inputs=inputs)
            
            # Check if we have pydantic output
            if hasattr(result, 'pydantic') and result.pydantic:
                return result.pydantic
            
            # Try to parse from raw output
            raw_output = getattr(result, 'raw', None) or str(result)
            
            # Extract JSON from raw output
            import re
            json_match = re.search(r'\{.*\}', raw_output, re.DOTALL)
            if json_match:
                parsed = json.loads(json_match.group())
                # Ensure pseudocode is included
                if 'pseudocode' not in parsed or not parsed['pseudocode']:
                    parsed['pseudocode'] = inputs['pseudocode']
                return AssessmentResult(**parsed)
                
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            if attempt == max_attempts - 1:
                # Return fallback result
                return AssessmentResult(
                    score=0,
                    correct=False,
                    summary="Evaluation failed - manual review required",
                    misconceptions=[],
                    pseudocode=inputs['pseudocode']
                )
            time.sleep(2)
            
    raise RuntimeError("All evaluation attempts failed")

# -------------------------
#  Example runtime assembly & kickoff
# -------------------------
# Build input_data
input_data = {
    'problem': "Buatlah algoritma untuk mencetak angka 1 sampai 5 menggunakan perulangan.",
    'context_solution': """
        program CetakAngka
        kamus
            i : integer
        algoritma
            for i <- 1 to 5 do
                output(i)
            endfor
        endprogram
    """,
    'pseudocode': """        
        program CobaLoop
        kamus
            i : integer
        algoritma
            i = 1
            while i <= 5 do
                output(i)
                if i > 5 then
                    break
                endif
                i = i + 1
            endwhile
        endprogram""",
    'general_rubrication': """
        Start: 100
        - Salah Syntax (Assign pakai '=' bukan '<-'): -5 poin per kejadian
        - Logic Error (Loop logic aneh): -15 poin
        - Miskonsepsi terdeteksi: -10 poin
    """
}

# Select chunks (same as before)
relevant_syntax_ref = select_pseudocode_chunks(pseudocode_chunks, input_data['pseudocode'])
relevant_misconceptions = select_misconception_chunks(misconception_chunks, input_data['pseudocode'])

# Build Task descriptions with strict JSON requirement
task_style = Task(
    description=(
        "Analisis pseudocode siswa berikut dan RETURN ONLY JSON matching the schema.\n\n"
        "REFERENSI SYNTAX (chunk relevan):\n" + relevant_syntax_ref + "\n\n"
        "PSEUDOCODE SISWA:\n{pseudocode}\n\n"
        "RETURN FORMAT (MUST): JSON object with keys: violations (list), summary (str), score_deduction (int).\n"
        "Example:\n" + STYLE_JSON_EXAMPLE + "\n\n"
        "Return ONLY the JSON object. No extra prose."
    ),
    expected_output="Laporan detail pelanggaran style and JSON.",
    agent=style_checker,
    max_iter=1
)

task_logic = Task(
    description=(
        "Analisis logika pseudocode siswa dan RETURN ONLY JSON matching schema.\n\n"
        "REFERENSI MISKONSEPSI (relevant subset):\n" + relevant_misconceptions + "\n\n"
        "PROBLEM:\n{problem}\n\n"
        "SOLUSI REFERENSI:\n{context_solution}\n\n"
        "PSEUDOCODE SISWA:\n{pseudocode}\n\n"
        "RETURN FORMAT (MUST): JSON with keys: misconceptions_detected (list), summary (str), penalty (int).\n"
        "Example:\n" + LOGIC_JSON_EXAMPLE + "\n\n"
        "Return ONLY the JSON object. No extra prose."
    ),
    expected_output="Laporan logic JSON.",
    agent=logic_checker,
    max_iter=1
)

task_final = Task(
    description=(
        "Based on the completed style and logic analysis, create final assessment. "
        "Use the outputs from previous tasks to calculate final score and determine correctness. "
        "Starting score: 100 points. "
        "Deduct points for style violations and logic misconceptions. "
        "Include the original student pseudocode in the response."
    ),
    expected_output="Final AssessmentResult JSON with all required fields populated.",
    agent=scoring_supervisor,
    context=[task_style, task_logic],  # This will provide previous task outputs
    output_pydantic=AssessmentResult,
    max_iter=1
)

# Assemble Crew
crew = Crew(
    agents=[style_checker, logic_checker, scoring_supervisor],  # Include supervisor
    tasks=[task_style, task_logic, task_final],
    process=Process.sequential,  # Changed from hierarchical
    verbose=True,
    max_iter=1,  # Reduced iterations
    step_callback=step_callback
)

# Run the evaluation with retries for robustness
try:
    final_model = run_evaluation_with_retries(crew, input_data, max_attempts=3)
    print("\n=== FINAL RESULT (Pydantic) ===")
    print(final_model.model_dump_json(indent=2))
except Exception as e:
    print("Evaluation ultimately failed:", e)
    # fallback: write logs for human review, or mark for manual grading



>> RUN ATTEMPT 1/3


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: cd706631-4776-4107-9631-a1f156aaacdb                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Style & Syntax Auditor                                                                                  │
│                                                                                                                 │
│  Task: Analisis pseudocode siswa berikut dan RETURN ONLY JSON matching the schema.                              │
│                                                                                                                 │
│  REFERENSI SYNTAX (chunk relevan):                                                                              │
│  # Pseudocode dan Golang Dasar                                                                                  │
│                                                                                                                 │
│  ## Daftar Isi                                                                                                  │
│                                                                                                                 │
│  1. [Output](#1-output)                                                                                         │
│  2. [Variabel dan Tipe Data](#2-variabel-dan-tipe-data)                                                         │
│  3. [Konstanta](#3-konstanta)                                                                                   │
│  4. [Komentar](#4-komentar)                                                                                     │
│  5. [Input dan Output](#5-input-dan-output)                                                                     │
│  6. [Operator](#6-operator)                                                                                     │
│  7. [Perulangan](#7-perulangan)                                                                                 │
│  8. [Percabangan](#8-percabangan)                                                                               │
│  9. [Perulangan dan Percabangan](#9-perulangan-dan-percabangan)                                                 │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 2-variabel-dan-tipe-data                                                                                    │
│                                                                                                                 │
│  Saat mendeklarasikan variabel perlu dituliskan pula tipe data yang dapat ditampung oleh variabel itu. Tipe     │
│  data dasar (primitif) yang dikenal dalam bahasa pemrograman adalah numerik (bilangan bulat dan real), huruf    │
│  (string dan karakter) serta boolean. Cara mendeklarasikannya dapat dilihat pada Program 9-16, yaitu dengan     │
│  memberi tanda titik dua (:) di antara nama variabel dengan tipe datanya.                                       │
│                                                                                                                 │
│  Dalam pemrograman kita tidak boleh mendeklarasikan identifier, termasuk nama variabel, dengan menggunakan      │
│  keyword yang tersedia pada compiler. Berikut keyword dalam Golang yang tidak bisa dijadikan nama               │
│  (identifier): import, type, const, var, func, package, map, chan, struct, interface, if, else, for, range,     │
│  break, continue, goto, return, switch, case, select, default, fallthrough, defer, go.                          │
│                                                        

↺ AgentFinish from :

/home/ilham/Documents/python/crewai-vs-langgraph/.venv/lib/python3.11/site-packages/rich/live.py:256: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Scoring Supervisor                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│  "score": 82,                                                                                                   │
│  "correct": false,                                                                                              │
│  "summary": "The student's pseudocode contains style violations and logic misconceptions. The while loop uses   │
│  a premature break pattern, and there are assignment syntax errors and operator usage errors.",                 │
│  "misconceptions": ["While demon (WD)"],                                                                        │
│  "pseudocode": "if x > 5 then\n  y = 10\nelse\n  z = 20\nend if\nwhile x < 10 do\n  x = x + 1\n  if x == 7      │
│  then break end if\n  print(x)\nend while"                                                                      │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/home/ilham/Documents/python/crewai-vs-langgraph/.venv/lib/python3.11/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: e01e83a9-c17f-44e0-b818-616f00c55d36                                                                     │
│  Agent: Scoring Supervisor                                                                                      │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: cd706631-4776-4107-9631-a1f156aaacdb                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: {                                                                                                │
│  "score": 82,                                                                                                   │
│  "correct": false,                                                                                              │
│  "summary": "The student's pseudocode contains style violations and logic misconceptions. The while loop uses   │
│  a premature break pattern, and there are assignment syntax errors and operator usage errors.",                 │
│  "misconceptions": ["While demon (WD)"],                                                                        │
│  "pseudocode": "if x > 5 then\n  y = 10\nelse\n  z = 20\nend if\nwhile x < 10 do\n  x = x + 1\n  if x == 7      │
│  then break end if\n  print(x)\nend while"                                                                      │
│  }                                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

input:
- pseoudocode
- problem
- context solution
- general rubrication

agen:
- scoring supervisior
- sub-agent 1: style checker agent
- sub-agent 2: logic checker agent

output:
```json
{
  "score":"value_scorenya",
  "correct":"true/false",
  "summary":"some_summary_here",
  "Misconceptions":"misconceptionsnya apa",
  "pseudocode":"pseudocode_siswa"
}
```

Possible Prompt for Each Agent:
- zero-shot
- few-shot
- chain-of-thought

TO DO Voice:
- Mendesain multi-agent system dengan 3 agen di dalamnya yaitu Style Checker, Logic Checker, dan Scoring Supervisor. Gunakan iterasi ganjil dalam interaksi antar agent seperti 1,3,5,dst.
- Eksplorasi berbagai LLM (free resource and can be fine tuned) yang bisa digunakan ke dalam agent.
- Gunakan prompting zero-shot atau COT pada Style Checker dan Logic Checker (boleh prompting lain juga).
- Tech-Stacknya bisa pakai Langchain / Langgraph